# 🧠 Vezilka — LaBSE Semantic Filtering

This notebook applies **LaBSE (Language-Agnostic BERT Sentence Embeddings)** to compute
true cross-lingual semantic similarity between every Macedonian ↔ Albanian pair.

### Pipeline
1. Load original dataset
2. Apply heuristic text fixes (backtick → ë, hyphen rejoining)
3. Normalise confidence values (handle 0–1000 vs 0–1 scale)
4. Compute LaBSE semantic similarity for all pairs
5. Analyse the similarity distribution
6. Filter by semantic + heuristic thresholds
7. Export cleaned dataset

**Model**: `sentence-transformers/LaBSE` — 471M params, 109 languages, 768-dim embeddings.

In [12]:
# ─── Imports ───────────────────────────────────────────────
import os, re, sys, time, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

# Add project root to path
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd()))
if os.path.basename(PROJECT_ROOT) != 'pipeline_v1':
    PROJECT_ROOT = os.path.join(PROJECT_ROOT, 'pipeline_v1')
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print(f"Project root: {PROJECT_ROOT}")
print(f"Python: {sys.version}")

Project root: /Users/edirizvani/VezillkaPipelineExctractor/pipeline_v1
Python: 3.12.4 | packaged by Anaconda, Inc. | (main, Jun 18 2024, 10:07:17) [Clang 14.0.6 ]


In [2]:
# ─── Configuration ─────────────────────────────────────────
#
# Tune these thresholds to control quality vs quantity.
#
# SEMANTIC_THRESHOLD: pairs below this LaBSE cosine similarity
#                     are considered mistranslations and removed.
#                     0.50 = lenient, 0.70 = strict, 0.60 = balanced.
#
# HEURISTIC_WEIGHT:   how much the old confidence contributes to
#                     the blended score (0 = pure semantic, 1 = pure heuristic).

SEMANTIC_THRESHOLD  = 0.60    # minimum LaBSE cosine similarity
HEURISTIC_WEIGHT    = 0.30    # blend: 70% semantic + 30% old confidence

# Basic quality filters (applied BEFORE semantic scoring)
MIN_WORDS           = 3       # sentences shorter than this → remove
MAX_WORDS           = 250     # sentences longer than this  → remove
MIN_LENGTH_RATIO    = 0.25    # MK/SQ char ratio below this → remove
MAX_LENGTH_RATIO    = 4.0     # MK/SQ char ratio above this → remove

# Encoding / batch size for LaBSE
BATCH_SIZE          = 256
DEVICE              = None    # None = auto-detect (GPU if available)

# Paths
INPUT_TSV  = os.path.join(PROJECT_ROOT, 'data', 'export', 'vezilka_mk_sq.tsv')
OUTPUT_TSV = os.path.join(PROJECT_ROOT, 'data', 'export', 'vezilka_mk_sq_semantic.tsv')

print(f"Input:  {INPUT_TSV}")
print(f"Output: {OUTPUT_TSV}")
print(f"Semantic threshold: {SEMANTIC_THRESHOLD}")
print(f"Blend weights: {1 - HEURISTIC_WEIGHT:.0%} semantic + {HEURISTIC_WEIGHT:.0%} heuristic")

Input:  /Users/edirizvani/VezillkaPipelineExctractor/pipeline_v1/data/export/vezilka_mk_sq.tsv
Output: /Users/edirizvani/VezillkaPipelineExctractor/pipeline_v1/data/export/vezilka_mk_sq_semantic.tsv
Semantic threshold: 0.6
Blend weights: 70% semantic + 30% heuristic


## 1. Load Original Dataset

In [3]:
df = pd.read_csv(INPUT_TSV, sep='\t', dtype=str)
df.columns = df.columns.str.strip()

# Fill NaN with empty strings for text columns
for col in ['mk', 'sq']:
    df[col] = df[col].fillna('')

# Parse confidence as float
df['confidence'] = pd.to_numeric(df['confidence'], errors='coerce').fillna(0.0)

original_count = len(df)
print(f"✅ Loaded {original_count:,} pairs")
print(f"   Columns: {list(df.columns)}")
df.head(3)

✅ Loaded 100,897 pairs
   Columns: ['mk', 'sq', 'source', 'article_id', 'confidence', 'method']


,mk,sq,source,article_id,confidence,method
0,"увоз - извоз - Блатец - Виница , со земјоделск...",Republika e Maqedonisë hyri në borxh ndaj Bank...,5DEEADB1E363224EAF2DFA137F40E518,1,0.885,structural+gc_2-1
1,Со оваа одлука престанува правото на користење...,Afati për pagesës së kreditit prej nenit 1 të ...,5DEEADB1E363224EAF2DFA137F40E518,2,0.888,structural+gc_1-1
2,увоз - извоз - Блатец - Ви - ница .,Pas efikasitetit të kreditit do të pagohet pro...,5DEEADB1E363224EAF2DFA137F40E518,2,0.436,structural+gc_1-1


## 2. Confidence Normalisation

Some rows have confidence in [0–1], others in [0–1000].  
We normalise everything to [0, 1].

In [4]:
# ─── Inspect confidence distribution ──────────────────────
print("Confidence stats BEFORE normalisation:")
print(df['confidence'].describe())
print(f"\nValues > 1.0:  {(df['confidence'] > 1.0).sum():,}")
print(f"Values > 100:  {(df['confidence'] > 100).sum():,}")
print(f"Values > 500:  {(df['confidence'] > 500).sum():,}")
print(f"Values in [0,1]: {((df['confidence'] >= 0) & (df['confidence'] <= 1)).sum():,}")

Confidence stats BEFORE normalisation:
count    100897.000000
mean          0.858071
std           0.091433
min           0.250000
25%           0.846000
50%           0.884000
75%           0.910000
max           0.931000
Name: confidence, dtype: float64

Values > 1.0:  0
Values > 100:  0
Values > 500:  0
Values in [0,1]: 100,897


In [5]:
# ─── Normalise to [0, 1] ──────────────────────────────────
# Strategy: if value > 1, assume it's on 0–1000 scale and divide by 1000.

mask_above_1 = df['confidence'] > 1.0
if mask_above_1.any():
    print(f"Normalising {mask_above_1.sum():,} confidence values from [0–1000] to [0–1]…")
    df.loc[mask_above_1, 'confidence'] = df.loc[mask_above_1, 'confidence'] / 1000.0

# Clip to [0, 1] just in case
df['confidence'] = df['confidence'].clip(0, 1)

print("\nConfidence stats AFTER normalisation:")
print(df['confidence'].describe())


Confidence stats AFTER normalisation:
count    100897.000000
mean          0.858071
std           0.091433
min           0.250000
25%           0.846000
50%           0.884000
75%           0.910000
max           0.931000
Name: confidence, dtype: float64


## 3. Heuristic Text Fixes

Fix known PDF extraction artifacts before computing embeddings.

In [6]:
# ─── Fix backtick → ë in Albanian text ────────────────────
backtick_before = df['sq'].str.contains('`', na=False).sum()
df['sq'] = df['sq'].str.replace('`', 'ë', regex=False)
backtick_after = df['sq'].str.contains('`', na=False).sum()
print(f"Backtick fix (Albanian): {backtick_before} → {backtick_after} rows with backtick")

# ─── Fix hyphenated line breaks ───────────────────────────
# "Ар - мијата" → "Армијата", "për - mbushur" → "përmbushur"
hyp_pattern = re.compile(r'(\w)\s*-\s+(\w)')
for col in ['mk', 'sq']:
    before = df[col].str.contains(r'\w\s+-\s+\w', regex=True, na=False).sum()
    df[col] = df[col].apply(lambda x: hyp_pattern.sub(r'\1\2', x) if isinstance(x, str) else x)
    after = df[col].str.contains(r'\w\s+-\s+\w', regex=True, na=False).sum()
    print(f"Hyphen fix ({col}): {before} → {after} rows with broken hyphens")

# ─── Remove 32-char hex hashes leaked from PDF processing ─
hex_pattern = re.compile(r'[0-9a-f]{32,}', re.IGNORECASE)
for col in ['mk', 'sq']:
    has_hex = df[col].str.contains(hex_pattern, na=False).sum()
    df[col] = df[col].apply(lambda x: hex_pattern.sub('', x).strip() if isinstance(x, str) else x)
    print(f"Hex hash removal ({col}): {has_hex} rows cleaned")

# ─── Normalise whitespace ─────────────────────────────────
for col in ['mk', 'sq']:
    df[col] = df[col].str.replace(r'\s+', ' ', regex=True).str.strip()

print(f"\n✅ Text fixes applied to {len(df):,} rows")

Backtick fix (Albanian): 219 → 0 rows with backtick
Hyphen fix (mk): 39039 → 19 rows with broken hyphens
Hyphen fix (sq): 7540 → 1 rows with broken hyphens
Hex hash removal (mk): 0 rows cleaned
Hex hash removal (sq): 0 rows cleaned

✅ Text fixes applied to 100,897 rows


## 4. Basic Quality Filters (Pre-Semantic)

Remove obviously bad pairs before expensive embedding computation.

In [7]:
before = len(df)

# Remove empty pairs
df = df[(df['mk'].str.len() > 0) & (df['sq'].str.len() > 0)].copy()
print(f"Empty removal:     {before - len(df):,} removed")

# Word count filters
df['mk_words'] = df['mk'].str.split().str.len()
df['sq_words'] = df['sq'].str.split().str.len()

before2 = len(df)
df = df[
    (df['mk_words'] >= MIN_WORDS) & (df['sq_words'] >= MIN_WORDS) &
    (df['mk_words'] <= MAX_WORDS) & (df['sq_words'] <= MAX_WORDS)
].copy()
print(f"Word count filter: {before2 - len(df):,} removed (min={MIN_WORDS}, max={MAX_WORDS})")

# Length ratio filter
df['len_ratio'] = df['mk'].str.len() / df['sq'].str.len().replace(0, 1)

before3 = len(df)
df = df[
    (df['len_ratio'] >= MIN_LENGTH_RATIO) &
    (df['len_ratio'] <= MAX_LENGTH_RATIO)
].copy()
print(f"Length ratio filter: {before3 - len(df):,} removed (ratio {MIN_LENGTH_RATIO}–{MAX_LENGTH_RATIO})")

# Script validation: MK should be mostly Cyrillic, SQ mostly Latin
def cyrillic_fraction(text):
    if not text: return 0
    letters = [c for c in text if c.isalpha()]
    if not letters: return 0
    return sum(1 for c in letters if '\u0400' <= c <= '\u04FF') / len(letters)

def latin_fraction(text):
    if not text: return 0
    letters = [c for c in text if c.isalpha()]
    if not letters: return 0
    return sum(1 for c in letters if ('A' <= c <= 'Z') or ('a' <= c <= 'z') or c in 'ëçÇËÜü') / len(letters)

df['mk_cyrillic'] = df['mk'].apply(cyrillic_fraction)
df['sq_latin'] = df['sq'].apply(latin_fraction)

before4 = len(df)
df = df[(df['mk_cyrillic'] > 0.5) & (df['sq_latin'] > 0.5)].copy()
print(f"Script validation:  {before4 - len(df):,} removed (wrong script)")

# Number-dominated sentences (> 40% digits)
def digit_fraction(text):
    if not text: return 0
    return sum(c.isdigit() for c in text) / max(len(text), 1)

before5 = len(df)
df = df[
    (df['mk'].apply(digit_fraction) < 0.4) &
    (df['sq'].apply(digit_fraction) < 0.4)
].copy()
print(f"Number filter:      {before5 - len(df):,} removed (>40% digits)")

# Exact duplicates
before6 = len(df)
df = df.drop_duplicates(subset=['mk', 'sq']).copy()
print(f"Exact dedup:        {before6 - len(df):,} removed")

print(f"\n✅ After pre-filters: {len(df):,} pairs (removed {original_count - len(df):,} total)")

# Clean up temp columns
df.drop(columns=['mk_words', 'sq_words', 'len_ratio', 'mk_cyrillic', 'sq_latin'], inplace=True, errors='ignore')

Empty removal:     0 removed
Word count filter: 5,333 removed (min=3, max=250)
Length ratio filter: 311 removed (ratio 0.25–4.0)
Script validation:  67 removed (wrong script)
Number filter:      459 removed (>40% digits)
Exact dedup:        1,693 removed

✅ After pre-filters: 93,034 pairs (removed 7,863 total)


## 5. LaBSE Semantic Similarity

Compute cosine similarity in the LaBSE embedding space for every pair.  
This is the **key step** — it replaces length-based heuristics with actual cross-lingual understanding.

In [9]:
from aligner.semantic_aligner import SemanticAligner

aligner = SemanticAligner(batch_size=BATCH_SIZE, device=DEVICE)

# Time the encoding
t0 = time.time()

mk_list = df['mk'].tolist()
sq_list = df['sq'].tolist()

print(f"Computing LaBSE embeddings for {len(mk_list):,} pairs…")
print(f"(batch_size={BATCH_SIZE}, device={DEVICE or 'auto'})")
print("This will take a few minutes on CPU…\n")

semantic_scores = aligner.score_pairs(mk_list, sq_list)

elapsed = time.time() - t0
print(f"\n⏱️  Done in {elapsed:.1f}s ({elapsed/60:.1f} min)")
print(f"   Speed: {len(mk_list)/elapsed:.0f} pairs/sec")

Computing LaBSE embeddings for 93,034 pairs…
(batch_size=256, device=auto)
This will take a few minutes on CPU…



ModuleNotFoundError: No module named 'sentence_transformers'

In [10]:
# ─── Store scores ─────────────────────────────────────────
df['semantic_score'] = semantic_scores

# Blended score: semantic + heuristic confidence
df['blended_score'] = SemanticAligner.blend_scores(
    semantic=df['semantic_score'].values,
    heuristic=df['confidence'].values,
    semantic_weight=1.0 - HEURISTIC_WEIGHT,
)

print("Semantic score distribution:")
print(df['semantic_score'].describe())
print(f"\nBlended score distribution:")
print(df['blended_score'].describe())

NameError: name 'semantic_scores' is not defined

## 6. Semantic Score Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# ─── Histogram of semantic scores ─────────────────────────
ax = axes[0]
ax.hist(df['semantic_score'], bins=100, color='#3498db', alpha=0.8, edgecolor='white')
ax.axvline(SEMANTIC_THRESHOLD, color='red', linestyle='--', linewidth=2, label=f'Threshold = {SEMANTIC_THRESHOLD}')
ax.set_xlabel('LaBSE Cosine Similarity')
ax.set_ylabel('Count')
ax.set_title('Distribution of Semantic Similarity Scores')
ax.legend()
ax.grid(True, alpha=0.3)

# ─── Semantic vs Heuristic confidence ─────────────────────
ax = axes[1]
sample = df.sample(min(5000, len(df)), random_state=42)
ax.scatter(sample['confidence'], sample['semantic_score'], alpha=0.15, s=5, c='#2ecc71')
ax.axhline(SEMANTIC_THRESHOLD, color='red', linestyle='--', linewidth=1.5, label=f'Semantic = {SEMANTIC_THRESHOLD}')
ax.axvline(0.5, color='orange', linestyle='--', linewidth=1.5, label='Heuristic = 0.5')
ax.set_xlabel('Heuristic Confidence (normalised)')
ax.set_ylabel('LaBSE Semantic Similarity')
ax.set_title('Semantic vs Heuristic Confidence')
ax.legend(loc='lower right')
ax.grid(True, alpha=0.3)

# ─── Cumulative distribution ──────────────────────────────
ax = axes[2]
sorted_scores = np.sort(df['semantic_score'].values)
cdf = np.arange(1, len(sorted_scores) + 1) / len(sorted_scores)
ax.plot(sorted_scores, cdf, color='#e74c3c', linewidth=2)
ax.axvline(SEMANTIC_THRESHOLD, color='red', linestyle='--', linewidth=1.5)

# Show what percentage is kept at the threshold
pct_kept = (df['semantic_score'] >= SEMANTIC_THRESHOLD).mean() * 100
ax.axhline(1 - pct_kept/100, color='gray', linestyle=':', alpha=0.5)
ax.text(SEMANTIC_THRESHOLD + 0.02, 0.5, f'{pct_kept:.1f}% kept', fontsize=11, color='red')

ax.set_xlabel('LaBSE Cosine Similarity')
ax.set_ylabel('Cumulative Fraction')
ax.set_title('Cumulative Distribution of Semantic Scores')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(PROJECT_ROOT, 'data', 'semantic_score_analysis.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f"📊 Plot saved to data/semantic_score_analysis.png")

In [ ]:
# ─── Score buckets analysis ───────────────────────────────
buckets = [
    ('< 0.3 (very poor)',  0.0, 0.3),
    ('0.3–0.5 (poor)',     0.3, 0.5),
    ('0.5–0.6 (marginal)', 0.5, 0.6),
    ('0.6–0.7 (fair)',     0.6, 0.7),
    ('0.7–0.8 (good)',     0.7, 0.8),
    ('0.8–0.9 (very good)',0.8, 0.9),
    ('≥ 0.9 (excellent)',  0.9, 1.01),
]

print(f"{'Bucket':<25} {'Count':>8} {'Percent':>8}")
print('─' * 45)
for label, lo, hi in buckets:
    n = ((df['semantic_score'] >= lo) & (df['semantic_score'] < hi)).sum()
    pct = n / len(df) * 100
    marker = ' ← threshold' if lo <= SEMANTIC_THRESHOLD < hi else ''
    print(f"{label:<25} {n:>8,} {pct:>7.1f}%{marker}")

print(f"\n{'Total':<25} {len(df):>8,}")

## 7. Sample Inspection — Good vs Bad Pairs

In [11]:
# ─── Best pairs (highest semantic similarity) ─────────────
print("=" * 80)
print("🟢 TOP 10 — BEST PAIRS (highest LaBSE similarity)")
print("=" * 80)

top = df.nlargest(10, 'semantic_score')
for i, (_, row) in enumerate(top.iterrows(), 1):
    print(f"\n#{i} — Score: {row['semantic_score']:.4f}")
    print(f"  MK: {row['mk'][:120]}")
    print(f"  SQ: {row['sq'][:120]}")

🟢 TOP 10 — BEST PAIRS (highest LaBSE similarity)


KeyError: 'semantic_score'

In [ ]:
# ─── Worst pairs (lowest semantic similarity) ─────────────
print("=" * 80)
print("🔴 BOTTOM 10 — WORST PAIRS (lowest LaBSE similarity)")
print("=" * 80)

bottom = df.nsmallest(10, 'semantic_score')
for i, (_, row) in enumerate(bottom.iterrows(), 1):
    print(f"\n#{i} — Score: {row['semantic_score']:.4f}")
    print(f"  MK: {row['mk'][:120]}")
    print(f"  SQ: {row['sq'][:120]}")

In [ ]:
# ─── Borderline pairs (near the threshold) ────────────────
print("=" * 80)
print(f"🟡 BORDERLINE PAIRS (semantic score near {SEMANTIC_THRESHOLD})")
print("=" * 80)

borderline = df[
    (df['semantic_score'] >= SEMANTIC_THRESHOLD - 0.05) &
    (df['semantic_score'] <= SEMANTIC_THRESHOLD + 0.05)
].sample(min(10, len(df)), random_state=42)

for i, (_, row) in enumerate(borderline.iterrows(), 1):
    print(f"\n#{i} — Score: {row['semantic_score']:.4f}")
    print(f"  MK: {row['mk'][:120]}")
    print(f"  SQ: {row['sq'][:120]}")

## 8. Apply Semantic Filter

In [ ]:
# ─── Filter by semantic threshold ─────────────────────────
df_clean = df[df['semantic_score'] >= SEMANTIC_THRESHOLD].copy()
df_rejected = df[df['semantic_score'] < SEMANTIC_THRESHOLD].copy()

print(f"✅ Semantic filter (threshold = {SEMANTIC_THRESHOLD}):")
print(f"   Kept:     {len(df_clean):>8,} ({len(df_clean)/len(df)*100:.1f}%)")
print(f"   Rejected: {len(df_rejected):>8,} ({len(df_rejected)/len(df)*100:.1f}%)")
print(f"   Original: {original_count:>8,}")
print(f"   Final:    {len(df_clean):>8,} ({len(df_clean)/original_count*100:.1f}% of original)")

## 9. Quality Comparison — Before vs After

In [ ]:
# ─── Compare original vs semantic-cleaned dataset ─────────
df_orig = pd.read_csv(INPUT_TSV, sep='\t', dtype=str)
df_orig['confidence'] = pd.to_numeric(df_orig['confidence'], errors='coerce').fillna(0)

metrics = {
    'Metric': [
        'Total pairs',
        'Mean MK length (chars)',
        'Mean SQ length (chars)',
        'Mean MK words',
        'Mean SQ words',
        'Mean heuristic confidence',
        'Mean semantic score',
        'Median semantic score',
        'Pairs with score > 0.8',
        'Unique MK sentences',
        'Unique SQ sentences',
    ],
    'Original': [
        f"{len(df_orig):,}",
        f"{df_orig['mk'].str.len().mean():.0f}",
        f"{df_orig['sq'].str.len().mean():.0f}",
        f"{df_orig['mk'].str.split().str.len().mean():.1f}",
        f"{df_orig['sq'].str.split().str.len().mean():.1f}",
        f"{df_orig['confidence'].mean():.3f}",
        'N/A',
        'N/A',
        'N/A',
        f"{df_orig['mk'].nunique():,}",
        f"{df_orig['sq'].nunique():,}",
    ],
    'Semantic-Cleaned': [
        f"{len(df_clean):,}",
        f"{df_clean['mk'].str.len().mean():.0f}",
        f"{df_clean['sq'].str.len().mean():.0f}",
        f"{df_clean['mk'].str.split().str.len().mean():.1f}",
        f"{df_clean['sq'].str.split().str.len().mean():.1f}",
        f"{df_clean['confidence'].mean():.3f}",
        f"{df_clean['semantic_score'].mean():.3f}",
        f"{df_clean['semantic_score'].median():.3f}",
        f"{(df_clean['semantic_score'] > 0.8).sum():,}",
        f"{df_clean['mk'].nunique():,}",
        f"{df_clean['sq'].nunique():,}",
    ],
}

comparison = pd.DataFrame(metrics)
comparison

In [ ]:
# ─── Visual comparison ────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# MK sentence length distribution
ax = axes[0]
ax.hist(df_orig['mk'].str.len(), bins=80, alpha=0.5, label=f'Original ({len(df_orig):,})', color='#e74c3c', density=True)
ax.hist(df_clean['mk'].str.len(), bins=80, alpha=0.5, label=f'Semantic ({len(df_clean):,})', color='#2ecc71', density=True)
ax.set_xlabel('MK Sentence Length (chars)')
ax.set_ylabel('Density')
ax.set_title('MK Sentence Length — Before vs After')
ax.set_xlim(0, 500)
ax.legend()
ax.grid(True, alpha=0.3)

# SQ sentence length distribution
ax = axes[1]
ax.hist(df_orig['sq'].str.len(), bins=80, alpha=0.5, label=f'Original ({len(df_orig):,})', color='#e74c3c', density=True)
ax.hist(df_clean['sq'].str.len(), bins=80, alpha=0.5, label=f'Semantic ({len(df_clean):,})', color='#2ecc71', density=True)
ax.set_xlabel('SQ Sentence Length (chars)')
ax.set_ylabel('Density')
ax.set_title('SQ Sentence Length — Before vs After')
ax.set_xlim(0, 500)
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(PROJECT_ROOT, 'data', 'semantic_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f"📊 Comparison plot saved to data/semantic_comparison.png")

## 10. Export New Dataset

In [ ]:
# ─── Prepare export columns ───────────────────────────────
export_cols = ['mk', 'sq', 'source', 'article_id', 'confidence', 'method', 'semantic_score', 'blended_score']

# Only keep columns that exist
export_cols = [c for c in export_cols if c in df_clean.columns]

# Sort by blended score descending (best pairs first)
df_export = df_clean[export_cols].sort_values('blended_score', ascending=False).reset_index(drop=True)

# Round score columns for readability
for col in ['confidence', 'semantic_score', 'blended_score']:
    if col in df_export.columns:
        df_export[col] = df_export[col].round(4)

print(f"Export shape: {df_export.shape}")
print(f"Columns: {list(df_export.columns)}")
df_export.head()

In [ ]:
# ─── Save to TSV ──────────────────────────────────────────
df_export.to_csv(OUTPUT_TSV, sep='\t', index=False)

file_size_mb = os.path.getsize(OUTPUT_TSV) / 1024 / 1024
print(f"\n✅ Semantic-cleaned dataset saved!")
print(f"   Path:  {OUTPUT_TSV}")
print(f"   Rows:  {len(df_export):,}")
print(f"   Size:  {file_size_mb:.1f} MB")
print(f"   Cols:  {list(df_export.columns)}")

In [ ]:
# ─── Also save a rejected pairs file for inspection ───────
rejected_path = os.path.join(PROJECT_ROOT, 'data', 'export', 'rejected_pairs.tsv')
reject_cols = [c for c in ['mk', 'sq', 'source', 'article_id', 'confidence', 'method', 'semantic_score'] if c in df_rejected.columns]

df_rejected[reject_cols].sort_values('semantic_score', ascending=True).to_csv(
    rejected_path, sep='\t', index=False
)

print(f"🗑️  Rejected pairs saved to: {rejected_path}")
print(f"   Rows: {len(df_rejected):,}")

## 11. Final Summary

In [ ]:
# ─── Summary ──────────────────────────────────────────────
print("╔" + "═" * 60 + "╗")
print("║" + " Vezilka — Semantic Filtering Summary".ljust(60) + "║")
print("╠" + "═" * 60 + "╣")
print(f"║ Original dataset:        {original_count:>8,} pairs".ljust(61) + "║")
print(f"║ After pre-filters:       {len(df):>8,} pairs".ljust(61) + "║")
print(f"║ After LaBSE (≥{SEMANTIC_THRESHOLD}):    {len(df_clean):>8,} pairs".ljust(61) + "║")
print(f"║ Rejected:                {len(df_rejected):>8,} pairs".ljust(61) + "║")
print(f"║ Retention rate:          {len(df_clean)/original_count*100:>7.1f}%".ljust(61) + "║")
print("╠" + "═" * 60 + "╣")
print(f"║ Model:                   LaBSE (768-dim)".ljust(61) + "║")
print(f"║ Threshold:               {SEMANTIC_THRESHOLD}".ljust(61) + "║")
print(f"║ Blend:                   {(1-HEURISTIC_WEIGHT)*100:.0f}% semantic + {HEURISTIC_WEIGHT*100:.0f}% heuristic".ljust(61) + "║")
print(f"║ Mean semantic score:     {df_clean['semantic_score'].mean():.4f}".ljust(61) + "║")
print(f"║ Median semantic score:   {df_clean['semantic_score'].median():.4f}".ljust(61) + "║")
print("╠" + "═" * 60 + "╣")
print(f"║ Output: {os.path.basename(OUTPUT_TSV)}".ljust(61) + "║")
print(f"║ Rejects: rejected_pairs.tsv".ljust(61) + "║")
print("╚" + "═" * 60 + "╝")